# Gesture Control Sandbox — welding pipeline

Turn hand gestures (laptop webcam) into one-shot commands for the perception
pipeline, and turn a pointing gesture into a SAM2 click coordinate.

**Key idea for pointing.** The webcam and the RealSense are different cameras
with no fixed geometry between them, so you can't map a fingertip in webcam
pixels to a RealSense pixel. Instead, you point at the part *as shown on a
displayed scene image*, using your index fingertip as a virtual cursor, and
latch a pixel `(u, v)`. That coordinate feeds `fp_server.py`'s `click` field.

In this sandbox the **webcam feed stands in for the RealSense RGB feed** so you
can experiment with nothing wired up. Swap it for the real feed later (see the
last section).

Pipeline of this notebook:

    webcam frame
       -> GestureRecognizer (7 built-in gestures)
       -> GestureLatch  (debounce + cooldown -> clean one-shot events)
       -> dispatch table (prints the ROS command; real calls are stubbed)

    index fingertip -> cursor over the frame -> dwell -> latched (u, v) click


## 0. Install + download the gesture model

Run once.

In [2]:
# %pip install mediapipe opencv-python numpy
# (uncomment the line above the first time)

import urllib.request, os

MODEL_PATH = "gesture_recognizer.task"
MODEL_URL = ("https://storage.googleapis.com/mediapipe-models/"
             "gesture_recognizer/gesture_recognizer/float16/1/"
             "gesture_recognizer.task")

if not os.path.exists(MODEL_PATH):
    print("downloading gesture model ...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
print("model ready:", os.path.abspath(MODEL_PATH))


model ready: /workspaces/welding_cell_ws/ros2_ws/src/admittance_control/notebooks/gesture_recognizer.task


## 1. Build the recognizer

VIDEO mode is the simplest for a webcam loop: one synchronous
`recognize_for_video(image, timestamp_ms)` call per frame. (LIVE_STREAM mode
exists too but needs an async callback — overkill for a sandbox.)

The model emits one of:
`None, Closed_Fist, Open_Palm, Pointing_Up, Thumb_Down, Thumb_Up, Victory, ILoveYou`.

In [3]:
import mediapipe as mp

BaseOptions            = mp.tasks.BaseOptions
GestureRecognizer      = mp.tasks.vision.GestureRecognizer
GestureRecognizerOptions = mp.tasks.vision.GestureRecognizerOptions
VisionRunningMode      = mp.tasks.vision.RunningMode

def make_recognizer(path=MODEL_PATH):
    opts = GestureRecognizerOptions(
        base_options=BaseOptions(model_asset_path=path),
        running_mode=VisionRunningMode.VIDEO,
        num_hands=1,                 # one operator hand
        min_hand_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    return GestureRecognizer.create_from_options(opts)

print("recognizer factory ready")


recognizer factory ready


## 2. Command mapping + the debounce/cooldown state machine

Two problems with raw per-frame gestures: they flicker during transitions, and
a held gesture fires 30x/second. `GestureLatch` fixes both:

- a gesture must be held for `stable_frames` consecutive frames before it counts
- it fires **once** on the rising edge into a new stable gesture
- it won't re-fire until the hand returns to neutral (`None`/unmapped) **and**
  `cooldown_s` has elapsed

Edit `GESTURE_TO_ACTION` to taste. `None` and anything unmapped are neutral.

In [ ]:
import time
from dataclasses import dataclass, field

# ---- map the 7 built-in gestures to your pipeline actions -------------------
GESTURE_TO_ACTION = {
    "ILoveYou":     "TRIGGER_BRIDGE",
    "Pointing_Up":  "INDICATE",   # enter cursor mode / move the SAM2 cursor
    "Thumb_Up":     "SEGMENT",    # confirm latched point -> bridge+SAM2 click
    "Closed_Fist":  "RUN_ICP",
    "Open_Palm":    "STOP_ICP",
    "Victory":      "SAVE_OBJECT",
    "Thumb_Down":   "CANCEL",
}
NEUTRAL = {None, "None"}

@dataclass
class GestureLatch:
    stable_frames: int = 6
    cooldown_s: float = 1.2
    _run_gesture: str | None = None
    _run_count: int = 0
    _armed: bool = True            # True when hand has passed through neutral
    _last_fire_t: float = 0.0

    def update(self, raw: str | None):
        # Feed the top gesture each frame. Returns an action string on a
        # clean rising edge, else None.
        # count consecutive identical raw frames
        if raw == self._run_gesture:
            self._run_count += 1
        else:
            self._run_gesture = raw
            self._run_count = 1

        # returning to neutral re-arms the latch
        if raw in NEUTRAL:
            self._armed = True
            return None

        action = GESTURE_TO_ACTION.get(raw)
        if action is None:
            return None  # unmapped gesture = neutral-ish, ignore

        stable = self._run_count >= self.stable_frames
        cooled = (time.time() - self._last_fire_t) >= self.cooldown_s
        if stable and self._armed and cooled:
            self._armed = False           # must pass through neutral before next fire
            self._last_fire_t = time.time()
            return action
        return None

print("GestureLatch ready")


GestureLatch ready


In [5]:
import subprocess

# ---- what each action actually does. STUBBED: prints only. -----------------
# Flip DRY_RUN to False (and you're on the ROS machine) to really call them.
DRY_RUN = True

ROS_CMDS = {
    "TRIGGER_BRIDGE": ["ros2","service","call","/foundationpose_bridge/trigger","std_srvs/srv/Trigger"],
    "RUN_ICP":        ["ros2","service","call","/icp_pose_refiner/run_icp","std_srvs/srv/Trigger"],
    "STOP_ICP":       ["ros2","service","call","/icp_pose_refiner/stop_tracking","std_srvs/srv/Trigger"],
    "SAVE_OBJECT":    ["ros2","service","call","/icp_pose_refiner/save_object","std_srvs/srv/Trigger"],
}

def dispatch(action, ctx=None):
    ctx = ctx or {}
    ts = time.strftime("%H:%M:%S")
    if action in ROS_CMDS:
        cmd = ROS_CMDS[action]
        print(f"[{ts}] {action:14s} -> {' '.join(cmd)}")
        if not DRY_RUN:
            subprocess.Popen(cmd)          # non-blocking; bridge trigger blocks a while
    elif action == "SEGMENT":
        uv = ctx.get("click_uv")
        print(f"[{ts}] SEGMENT        -> SAM2 click at {uv}  (send in fp_server 'click' field)")
        # not DRY_RUN: POST rgb/depth/K + click=uv to fp_server, then TRIGGER_BRIDGE
    elif action == "INDICATE":
        print(f"[{ts}] INDICATE       -> cursor mode (handled in the loop)")
    elif action == "CANCEL":
        print(f"[{ts}] CANCEL         -> clear latched point / abort")
    else:
        print(f"[{ts}] {action}  (no handler)")

print("dispatch ready  (DRY_RUN =", DRY_RUN, ")")


dispatch ready  (DRY_RUN = True )


## 3. Live loop — gestures to events

Run this cell. A window opens; hold a gesture until it fires (watch the
printout). Press **q** to quit.

If the OpenCV window misbehaves inside Jupyter (freezes, won't close), copy this
cell's code into a `.py` file and run it from a terminal — the code is identical
and behaves better outside the notebook event loop.

In [6]:
import cv2, numpy as np

def gesture_loop():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("no webcam at index 0")
    latch = GestureLatch(stable_frames=6, cooldown_s=1.2)
    t0 = time.time()
    try:
        with make_recognizer() as rec:
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                frame = cv2.flip(frame, 1)                      # mirror = intuitive
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
                ts_ms = int((time.time() - t0) * 1000)
                res = rec.recognize_for_video(mp_img, ts_ms)

                top = None
                if res.gestures:
                    g = res.gestures[0][0]
                    top = g.category_name
                    cv2.putText(frame, f"{top}  {g.score:.2f}", (10, 40),
                                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)

                action = latch.update(top)
                if action:
                    dispatch(action)

                cv2.imshow("gesture sandbox  (q to quit)", frame)
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    break
    finally:
        cap.release()
        cv2.destroyAllWindows()

gesture_loop()


I0000 00:00:1785151241.541245   14858 hand_gesture_recognizer_graph.cc:250] Custom gesture classifier is not defined.
I0000 00:00:1785151241.613428   14858 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1785151241.619281   14886 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.2), renderer: AMD Radeon 780M Graphics (radeonsi, phoenix, LLVM 20.1.2, DRM 3.57, 6.8.0-136-generic)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1785151241.647254   14865 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1785151241.662238   14873 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1785151241.666526   14871 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature infere

[11:20:45] RUN_ICP        -> ros2 service call /icp_pose_refiner/run_icp std_srvs/srv/Trigger
[11:20:46] RUN_ICP        -> ros2 service call /icp_pose_refiner/run_icp std_srvs/srv/Trigger
[11:20:53] STOP_ICP       -> ros2 service call /icp_pose_refiner/stop_tracking std_srvs/srv/Trigger
[11:20:54] RUN_ICP        -> ros2 service call /icp_pose_refiner/run_icp std_srvs/srv/Trigger
[11:20:56] CANCEL         -> clear latched point / abort
[11:21:06] TRIGGER_BRIDGE -> ros2 service call /foundationpose_bridge/trigger std_srvs/srv/Trigger
[11:21:08] STOP_ICP       -> ros2 service call /icp_pose_refiner/stop_tracking std_srvs/srv/Trigger
[11:21:10] STOP_ICP       -> ros2 service call /icp_pose_refiner/stop_tracking std_srvs/srv/Trigger
[11:21:12] RUN_ICP        -> ros2 service call /icp_pose_refiner/run_icp std_srvs/srv/Trigger
[11:21:14] TRIGGER_BRIDGE -> ros2 service call /foundationpose_bridge/trigger std_srvs/srv/Trigger
[11:21:18] TRIGGER_BRIDGE -> ros2 service call /foundationpose_bridge

## 4. Pointing — fingertip cursor + dwell-to-latch

Now the SAM2 click. Index-fingertip landmark (id 8) is normalized to the frame,
so it maps **directly** to a pixel — no calibration. Hold the fingertip still
for `dwell_s` and the point latches (turns red); that's your `(u, v)` click.

Gestures still fire in parallel, so in practice: point (`Pointing_Up`), let the
cursor settle to latch, then `Thumb_Up` -> `SEGMENT` sends the latched pixel.

Landmarks are in the **displayed image's** coordinates, which is exactly what
SAM2 wants — as long as this frame is the same one you send to the server. In
the real system, draw the cursor on the RealSense RGB, not the webcam.

In [7]:
def cursor_loop():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("no webcam at index 0")
    latch = GestureLatch(stable_frames=6, cooldown_s=1.2)

    # dwell state
    dwell_s = 0.8
    move_tol = 18            # px; movement under this counts as "holding still"
    hover_uv = None
    hover_since = None
    latched_uv = None
    t0 = time.time()

    try:
        with make_recognizer() as rec:
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                frame = cv2.flip(frame, 1)
                h, w = frame.shape[:2]
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
                ts_ms = int((time.time() - t0) * 1000)
                res = rec.recognize_for_video(mp_img, ts_ms)

                top = None
                tip_uv = None
                if res.gestures:
                    top = res.gestures[0][0].category_name
                if res.hand_landmarks:
                    lm = res.hand_landmarks[0][8]           # index fingertip
                    tip_uv = (int(lm.x * w), int(lm.y * h))

                # ---- dwell logic (only while indicating) ----
                indicating = (top == "Pointing_Up")
                if indicating and tip_uv is not None:
                    cv2.circle(frame, tip_uv, 10, (0, 255, 255), 2)   # live cursor
                    if hover_uv is None or (abs(tip_uv[0]-hover_uv[0]) > move_tol or
                                            abs(tip_uv[1]-hover_uv[1]) > move_tol):
                        hover_uv = tip_uv
                        hover_since = time.time()
                    elif time.time() - hover_since >= dwell_s:
                        latched_uv = tip_uv
                else:
                    hover_uv, hover_since = None, None

                if latched_uv is not None:
                    cv2.circle(frame, latched_uv, 8, (0, 0, 255), -1)  # latched click
                    cv2.putText(frame, f"click {latched_uv}", (10, h-20),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

                if top:
                    cv2.putText(frame, top, (10, 40),
                                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)

                # ---- fire actions, passing the latched click as context ----
                action = latch.update(top)
                if action == "CANCEL":
                    latched_uv = None
                    dispatch(action)
                elif action:
                    dispatch(action, ctx={"click_uv": latched_uv})

                cv2.imshow("pointing sandbox  (q to quit)", frame)
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    break
    finally:
        cap.release()
        cv2.destroyAllWindows()

cursor_loop()


I0000 00:00:1785151334.964640   15268 hand_gesture_recognizer_graph.cc:250] Custom gesture classifier is not defined.
I0000 00:00:1785151334.968611   15268 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1785151334.970313   15289 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.2), renderer: AMD Radeon 780M Graphics (radeonsi, phoenix, LLVM 20.1.2, DRM 3.57, 6.8.0-136-generic)
W0000 00:00:1785151334.996050   15275 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1785151335.011614   15280 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1785151335.012838   15279 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00

[11:22:36] INDICATE       -> cursor mode (handled in the loop)
[11:22:39] RUN_ICP        -> ros2 service call /icp_pose_refiner/run_icp std_srvs/srv/Trigger
[11:22:41] INDICATE       -> cursor mode (handled in the loop)
[11:22:42] RUN_ICP        -> ros2 service call /icp_pose_refiner/run_icp std_srvs/srv/Trigger
[11:22:44] INDICATE       -> cursor mode (handled in the loop)
[11:22:46] RUN_ICP        -> ros2 service call /icp_pose_refiner/run_icp std_srvs/srv/Trigger
[11:22:48] RUN_ICP        -> ros2 service call /icp_pose_refiner/run_icp std_srvs/srv/Trigger
[11:22:49] RUN_ICP        -> ros2 service call /icp_pose_refiner/run_icp std_srvs/srv/Trigger
[11:22:50] INDICATE       -> cursor mode (handled in the loop)
[11:22:52] RUN_ICP        -> ros2 service call /icp_pose_refiner/run_icp std_srvs/srv/Trigger
[11:22:53] INDICATE       -> cursor mode (handled in the loop)
[11:22:55] RUN_ICP        -> ros2 service call /icp_pose_refiner/run_icp std_srvs/srv/Trigger
[11:22:56] INDICATE       -

## 5. Wiring it into the real pipeline

**Swap the feed.** Replace `cv2.VideoCapture(0)` with the RealSense RGB the
laptop already has (the bridge captures it). Draw the cursor on *that* image so
the latched `(u, v)` is in RealSense pixels — the frame you draw on must be the
frame you send to the server.

**Send the click.** `fp_server.py` accepts a `click` field to skip its popup.
On `SEGMENT`, POST the current rgb/depth/K plus `click=latched_uv`; SAM2 masks
from that point, PPF identifies the CAD, FoundationPose registers. This deletes
the host-side click entirely.

**Real dispatch.** Set `DRY_RUN = False` on the ROS machine. Prefer swapping the
`subprocess` calls for `rclpy` service clients once you're past sandboxing —
sharing one node avoids spawning a process per gesture and gives you real
success/failure back instead of fire-and-forget.

**Suggested full sequence, all by gesture:**

    Pointing_Up (dwell to latch)  ->  Thumb_Up  ->  SEGMENT  (bridge + SAM2 + PPF + FP)
    Closed_Fist  ->  RUN_ICP        (seed + track)
    Open_Palm    ->  STOP_ICP
    Victory      ->  SAVE_OBJECT     (bake into SEPC, next part)
    Thumb_Down   ->  CANCEL          (clear latch / abort)

**Reliability knobs.** `stable_frames` up = fewer misfires, more lag.
`cooldown_s` guards against accidental repeats. For `SAVE_OBJECT` (destructive)
consider a two-gesture confirm, or reuse the operator's existing habit of
confirming the PPF winner in the popup.

**Beyond the 7 gestures.** If you need more distinct commands, MediaPipe Model
Maker trains a custom `.task` from your own labelled hand images — same runtime
API, just a different model file. Only worth it once the built-in set runs out.
